In [5]:
import os
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import keyring
from io import StringIO
import requests
from matplotlib.ticker import FuncFormatter, MaxNLocator

import eikon as ek
from pydatastream import Datastream

from scipy.stats import ttest_ind, ttest_ind_from_stats
from scipy.special import stdtr

# Connection to Datastream

In [6]:
# --- Log ind ---
ek.set_app_key('035d2f1682244553ba7f239ce9e8d281142013dd')
service = "Datastream"
DS_USERNAME = "ZNYK749" #Ændre denne
DS_PASSWORD = keyring.get_password(service, DS_USERNAME)

if DS_USERNAME and DS_PASSWORD:
    DS = Datastream(username=DS_USERNAME, password=DS_PASSWORD)
else:
    DS = None
    print("WARNING: DS_USERNAME/DS_PASSWORD ikke sat som miljøvariabler.")

# Dates

In [7]:
pd.set_option('display.max_rows', 10)

start = '20001231'
end = '20251231'
start_ds = start
end_ds = end

source = 'datastream'
fx = 'DKK'
width_bar = 1

N = 1
ind = np.arange(N)

# Index

In [20]:
# Define a helper function to simplify fetching and renaming data
def fetch_and_rename_data(symbol, column_name):
    data = DS.get_price([symbol], date_from=start_ds, date_to=end_ds)
    data = data.rename(columns={'P': column_name})
    return data

# Fetch and merge equity data
data = pd.DataFrame(index=pd.date_range(start_ds, end_ds, freq='B'))

equity_data = [
    ('MSWRLD$(MSNR)~DKK', 'MSCI World'),
    ('MSNAMR$(MSNR)~DKK', 'MSCI North America'),
    ('MSEROP$(MSNR)~DKK', 'MSCI Europe'),
    ('MSDNMKL(MSNR)', 'MSCI Denmark'),
    ('MSJPANL(MSNR)~DKK', 'MSCI Japan'),
    ('MSEMKF$(MSNR)~DKK', 'MSCI EM'),
    ('MSACWF$(MSNR)~DKK', 'MSCI ACWI'),
    ('M1AFCS$(MSNR)~DKK', 'MSCI World Consumer Staples'),
    ('M1AFID$(MSNR)~DKK', 'MSCI World Industrials'),
    ('M1AFHC$(MSNR)~DKK', 'MSCI World Health Care'),
    ('M1AFM1$(MSNR)~DKK', 'MSCI World Materials'),
    ('M1AFIT$(MSNR)~DKK', 'MSCI World IT'),
    ('M1AFFN$(MSNR)~DKK', 'MSCI World Financials'),
    ('M1AFCD$(MSNR)~DKK', 'MSCI World Consumer Discretionary'),
    ('M1AFE1$(MSNR)~DKK', 'MSCI World Energy'),
    ('M1AFR1$(MSNR)~DKK', 'MSCI World Real Estate'),
    ('M1AFU1$(MSNR)~DKK', 'MSCI World Utilities'),
    ('M1AFT1$(MSNR)~DKK', 'MSCI World Communication Services'),
    ('MSGWLD$(MSNR)~DKK', 'MSCI World Growth'),
    ('MSVWLD$(MSNR)~DKK', 'MSCI World Value'),
    ('MSDWQA$(MSNR)~DKK', 'MSCI World Quality'),
    ('MSWDMV$(MSNR)~DKK', 'MSCI World Minimum Volatility'),
    ('MHDYDW$(MSNR)~DKK', 'MSCI World High Dividend Yield'),
    ('MSDWST$(MSNR)~DKK', 'MSCI World Size Tilt'),
    ('MSSUSA$(MSNR)~DKK', 'MSCI USA Small Cap'),
    ('MSLUSA$(MSNR)~DKK', 'MSCI USA Large Cap'),
    ('MSDWCL$(MSNR)~DKK', 'MSCI World Cyclicals'),
    ('MSDWDF$(MSNR)~DKK', 'MSCI World Defensives')
]

for symbol, column_name in equity_data:
    equity = fetch_and_rename_data(symbol, column_name)
    data = pd.merge(data, equity, left_index=True, right_index=True)

# Fetch and merge bond data

bond_data = [
    #('ERC0(ML:RI#T)', 'European IG'),
    #('HW00(ML:RIHDEUR)', 'Global HY'),
    ('NYKMTSH(RI)', 'Danish Mortgage Bonds Short Index'),
    ('NYKMTLM(RI)', 'Danish Mortgage Bonds Long Index'),
    ('NDGVT5Y(RI)', 'Nordea 5-year index'),
    ('NDGVT2Y(RI)', 'Nordea 2-year index'),
    ('NDGVT7Y(RI)', 'Nordea 7-year index'),
    ('BMUS10Y(RI)~DKK', 'US Treasuries')
]

for symbol, column_name in bond_data:
    bond = fetch_and_rename_data(symbol, column_name)
    data = pd.merge(data, bond, left_index=True, right_index=True)

# Fetch and merge additional data
ust_usd = fetch_and_rename_data('BMUS10Y(RI)', 'US Treasuries in USD')
min_vol = fetch_and_rename_data('(MSDWME$(MSNR))~DKK', 'MSCI World Min Vol')

data = pd.merge(data, ust_usd, left_index=True, right_index=True)
data = pd.merge(data, min_vol, left_index=True, right_index=True)


# Store equity data
equities = data.copy()
print(equities.head())

            MSCI World  MSCI North America  MSCI Europe  MSCI Denmark  \
2001-01-01   19776.354           23205.065    25970.882      4792.542   
2001-01-02   19280.783           22373.125    25613.007      4849.926   
2001-01-03   19723.245           23521.686    25317.568      4908.359   
2001-01-04   19747.559           23295.170    25801.644      4985.422   
2001-01-05   19356.346           22509.092    25664.580      5024.605   

            MSCI Japan  MSCI EM  MSCI ACWI  MSCI World Consumer Staples  \
2001-01-01      82.998  794.213    794.954                       794.83   
2001-01-02      82.009  787.681    775.706                       788.02   
2001-01-03      82.215  794.056    792.878                       769.30   
2001-01-04      81.532  823.016    795.260                       743.13   
2001-01-05      81.282  820.308    780.188                       737.98   

            MSCI World Industrials  MSCI World Health Care  ...  \
2001-01-01                  795.04         

# Futures

In [19]:
def fetch_futures_data(symbol, column_name):
    data = DS.get_price([symbol], date_from=start_ds, date_to=end_ds)
    data = data.rename(columns={'P': column_name})
    return data

# Start med en tom dataframe — ingen foruddefineret index
futures_data = None

equity_futures = [
    ('ISMCS00', 'S&P 500'),          # USD
    ('CENCS00', 'NASDAQ 100'),       # USD
    ('GEXCS00', 'Euro Stoxx 50'),    # EUR
    #('ONACS00', 'Nikkei 225'),      # JPY             #Af en eller anden årsag jeg ikke kender, så kan jeg ikke hive denne ud her i Datastream
    #('SCNCS00' , 'FTSE China A50'),  # CNY            #Af en eller anden årsag jeg ikke kender, så kan jeg ikke hive denne ud her i Datastream
    ('GDXCS00', 'DAX'),              # EUR
    ('HSICS00', 'Hang Seng'),         # HKD
    ('LSXCS00', 'FTSE 100'),         # GBP
    ('BSXCS00', 'India')              # Ekstra
]

bond_futures = [
    ('GGECS00', 'Euro Bund 10Y'),    # EUR
    ('GBECS00', 'EURO BOBL 5Y'),      # EUR
    ('GEBCS00', 'EURO SCHATZ 2Y'),    # EUR
    ('CTTCS00', 'US T-Note 10Y'),    # USD
    ('CTFCS00', 'US T-Note 5Y'),     # USD
    ('CTECS00', 'US T-Note 2Y')     # USD
]

for symbol, column_name in equity_futures + bond_futures:
    series = fetch_futures_data(symbol, column_name)
    if futures_data is None:
        futures_data = series
    else:
        futures_data = pd.merge(futures_data, series,
                                left_index=True, right_index=True,
                                how='outer')

print(futures_data.head())
print(futures_data.shape)

            S&P 500  NASDAQ 100  Euro Stoxx 50     DAX  Hang Seng  FTSE 100  \
2000-12-29   1354.9      2376.5         4812.0  6500.0       2292    6200.0   
2001-01-01   1354.9      2376.5         4812.0  6500.0       2292    6200.0   
2001-01-02   1331.6      2170.5         4738.0  6340.0       2292    6190.0   
2001-01-03   1294.0      2531.5         4650.0  6489.0       2292    6045.0   
2001-01-04   1355.7      2485.0         4820.0  6435.0       2292    6237.0   

             India  Euro Bund 10Y  EURO BOBL 5Y  EURO SCHATZ 2Y  \
2000-12-29  3880.3         108.45        106.04          102.78   
2001-01-01  3880.3         108.45        106.04          102.78   
2001-01-02  3880.3         109.34        106.53          102.94   
2001-01-03  4060.5         109.36        106.39          102.89   
2001-01-04  4167.5         109.24        106.37          103.00   

            US T-Note 10Y  US T-Note 5Y  US T-Note 2Y  
2000-12-29       104.8906      103.5781      101.5859  
2001-01-01

In [16]:
test_codes = [
    'ONACS00', #Nikkei
    'SCNCS00', #China
]

for code in test_codes:
    try:
        test = DS.get_price([code], date_from='2020-01-01', date_to='2023-01-01')
        print(f"✓ {code} virker — {len(test)} rækker")
    except Exception as e:
        print(f"✗ {code} fejler")

✗ ONACS00 fejler
✗ SCNCS00 fejler


# FX kurslister

In [24]:
fx_data_list = [
    ('USDDK.', 'USD_DKK'),
    ('EURDK.', 'EUR_DKK'),
    ('GBPDK.', 'GBP_DKK'),
    ('HKDDK.', 'HKD_DKK'),
    ('INDDK.', 'INR_DKK'),
]

fx_data = None
for symbol, column_name in fx_data_list:
    try:
        series = fetch_futures_data(symbol, column_name)
        if fx_data is None:
            fx_data = series
        else:
            fx_data = pd.merge(fx_data, series,
                               left_index=True, right_index=True,
                               how='outer')
        print(f"✓ {column_name}")
    except Exception as e:
        print(f"✗ {column_name} fejler — prøv anden kode")

✗ USD_DKK fejler — prøv anden kode
✗ EUR_DKK fejler — prøv anden kode
✗ GBP_DKK fejler — prøv anden kode
✗ HKD_DKK fejler — prøv anden kode
✗ INR_DKK fejler — prøv anden kode
